<a href="https://colab.research.google.com/github/192321178/Machine-Learning-ITA0604/blob/main/ML_EXP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import csv

rows = [
    ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same', 'Yes'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same', 'Yes'],
    ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change', 'No'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change', 'Yes']
]

header = ['Sky', 'AirTemp', 'Humidity', 'Wind',
          'Water', 'Forecast', 'EnjoySport']

with open('enjoysport.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(header)
    writer.writerows(rows)

with open('enjoysport.csv', newline='') as file:
    data = list(csv.reader(file))[1:]

hypothesis = ['0'] * (len(data[0]) - 1)

for row in data:
    if row[-1].strip().lower() == 'yes':
        for i, value in enumerate(row[:-1]):
            if hypothesis[i] == '0':
                hypothesis[i] = value
            elif hypothesis[i] != value:
                hypothesis[i] = '?'

print("Most specific hypothesis:", hypothesis)

Most specific hypothesis: ['Sunny', 'Warm', '?', 'Strong', '?', '?']


In [2]:
from itertools import product

X = [
    ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same'],
    ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change']
]

y = ['Yes', 'Yes', 'No', 'Yes']

domains = [
    sorted({row[j] for row in X})
    for j in range(len(X[0]))
]

hypotheses = list(
    product(*[values + ['?'] for values in domains])
)

def covers(h, example):
    return all(
        a == '?' or a == value
        for a, value in zip(h, example)
    )

def more_general(h1, h2):
    return all(
        a == '?' or a == b
        for a, b in zip(h1, h2)
    )

consistent = [
    h for h in hypotheses
    if all(
        covers(h, example) == (label == 'Yes')
        for example, label in zip(X, y)
    )
]

S = [
    h for h in consistent
    if not any(
        h2 != h and more_general(h, h2)
        for h2 in consistent
    )
]

G = [
    h for h in consistent
    if not any(
        h2 != h and more_general(h2, h)
        for h2 in consistent
    )
]

print("Final S:", [list(h) for h in S])
print("Final G:", [list(h) for h in G])

Final S: [['Sunny', 'Warm', '?', 'Strong', '?', '?']]
Final G: [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?']]


In [3]:
from collections import Counter
from math import log2

data = [
    ('Sunny', 'Hot', 'High', 'Weak', 'No'),
    ('Sunny', 'Hot', 'High', 'Strong', 'No'),
    ('Overcast', 'Hot', 'High', 'Weak', 'Yes'),
    ('Rain', 'Mild', 'High', 'Weak', 'Yes'),
    ('Rain', 'Cool', 'Normal', 'Weak', 'Yes'),
    ('Rain', 'Cool', 'Normal', 'Strong', 'No'),
    ('Overcast', 'Cool', 'Normal', 'Strong', 'Yes'),
    ('Sunny', 'Mild', 'High', 'Weak', 'No'),
    ('Sunny', 'Cool', 'Normal', 'Weak', 'Yes'),
    ('Rain', 'Mild', 'Normal', 'Weak', 'Yes'),
    ('Sunny', 'Mild', 'Normal', 'Strong', 'Yes'),
    ('Overcast', 'Mild', 'High', 'Strong', 'Yes'),
    ('Overcast', 'Hot', 'Normal', 'Weak', 'Yes'),
    ('Rain', 'Mild', 'High', 'Strong', 'No')
]

attributes = ['Outlook', 'Temperature', 'Humidity', 'Wind']
target_index = 4

def entropy(rows):
    counts = Counter(row[target_index] for row in rows)
    total = len(rows)

    return -sum(
        (n / total) * log2(n / total)
        for n in counts.values()
    )

def information_gain(rows, attr_index):
    base = entropy(rows)
    values = sorted({row[attr_index] for row in rows})
    remainder = 0.0

    for value in values:
        subset = [
            row for row in rows
            if row[attr_index] == value
        ]

        remainder += (
            len(subset) / len(rows)
        ) * entropy(subset)

    return base - remainder

def id3(rows, attrs):
    labels = [row[target_index] for row in rows]

    if len(set(labels)) == 1:
        return labels[0]

    if not attrs:
        return Counter(labels).most_common(1)[0][0]

    best = max(
        attrs,
        key=lambda a: information_gain(
            rows, attributes.index(a)
        )
    )

    tree = {best: {}}

    for value in sorted({
        row[attributes.index(best)] for row in rows
    }):
        subset = [
            row for row in rows
            if row[attributes.index(best)] == value
        ]

        remaining = [a for a in attrs if a != best]

        tree[best][value] = id3(subset, remaining)

    return tree

def predict(tree, sample):
    if isinstance(tree, str):
        return tree

    attribute = next(iter(tree))
    value = sample[attribute]
    branch = tree[attribute].get(value)

    return 'Unknown' if branch is None else predict(branch, sample)

tree = id3(data, attributes)

sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

print("Root attribute:", next(iter(tree)))
print("Predicted class:", predict(tree, sample))

Root attribute: Outlook
Predicted class: No


In [4]:
import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

iris = load_iris()

X_train, X_test, y_train, y_test = train_test_split(
    iris.data,
    iris.target,
    test_size=0.30,
    random_state=1
)

model = DecisionTreeClassifier(
    criterion='entropy',
    random_state=1
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")

plt.figure(figsize=(12, 8))

plot_tree(
    model,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True
)

plt.tight_layout()
plt.savefig('iris_tree.png', dpi=150)

print("Decision tree saved as iris_tree.png")

Accuracy: 0.9556
Decision tree saved as iris_tree.png


In [5]:
import numpy as np

X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

y = np.array([0, 0, 0, 1])

weights = np.zeros(3)

learning_rate = 0.1

for _ in range(20):
    for inputs, target in zip(X, y):
        inputs_with_bias = np.insert(inputs, 0, 1)

        prediction = (
            1 if np.dot(weights, inputs_with_bias) > 0 else 0
        )

        weights += (
            learning_rate *
            (target - prediction) *
            inputs_with_bias
        )

print("Learned weights (bias, w1, w2):", weights)

for inputs in X:
    inputs_with_bias = np.insert(inputs, 0, 1)

    prediction = (
        1 if np.dot(weights, inputs_with_bias) > 0 else 0
    )

    print(f"{inputs.tolist()} -> {prediction}")

Learned weights (bias, w1, w2): [-0.2  0.2  0.1]
[0, 0] -> 0
[0, 1] -> 0
[1, 0] -> 0
[1, 1] -> 1


In [6]:
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score

X = np.array([
    [-2, 0],
    [-1, 1],
    [-1, -1],
    [0, 1],
    [1, 0],
    [1, 1],
    [2, 0],
    [2, 1]
], dtype=float)

y = (X[:, 0] > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=1,
    stratify=y
)

model = Perceptron(
    max_iter=1000,
    eta0=0.1,
    tol=1e-3,
    random_state=1
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, predictions):.2f}")
print("Predictions:", predictions)

Test accuracy: 1.00
Predictions: [0 1]


In [7]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(output):
    return output * (1 - output)

X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)

y = np.array([
    [0],
    [1],
    [1],
    [0]
], dtype=float)

np.random.seed(1)

w1 = np.random.uniform(-1, 1, (2, 4))
w2 = np.random.uniform(-1, 1, (4, 1))

for _ in range(10000):
    hidden = sigmoid(X @ w1)
    output = sigmoid(hidden @ w2)

    output_error = (
        (y - output) * sigmoid_derivative(output)
    )

    hidden_error = (
        (output_error @ w2.T) *
        sigmoid_derivative(hidden)
    )

    w2 += hidden.T @ output_error * 0.5
    w1 += X.T @ hidden_error * 0.5

print("Predicted output after training:")
print(np.round(output, 3))

Predicted output after training:
[[0.032]
 [0.975]
 [0.971]
 [0.028]]


In [8]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=1,
    stratify=y
)

scaler = StandardScaler().fit(X_train)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

model = MLPClassifier(
    hidden_layer_sizes=(10,),
    solver='lbfgs',
    max_iter=1000,
    random_state=1
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")

Accuracy: 0.9778


In [9]:
import tensorflow as tf

from tensorflow.keras import layers, models

tf.keras.utils.set_random_seed(7)

(X_train, y_train), (X_test, y_test) = (
    tf.keras.datasets.mnist.load_data()
)

X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

model = models.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    X_train,
    y_train,
    epochs=1,
    batch_size=128,
    verbose=0
)

print("MNIST training completed.")
print("Input shape:", model.input_shape)
print("Output classes:", model.output_shape[-1])

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
MNIST training completed.
Input shape: (None, 28, 28)
Output classes: 10


In [10]:
import random

random.seed(7)

def decode(chromosome):
    return int(chromosome, 2)

def fitness(chromosome):
    return decode(chromosome) ** 2

population = ['11111'] + [
    ''.join(random.choice('01') for _ in range(5))
    for _ in range(5)
]

for _ in range(20):
    scores = [fitness(c) for c in population]

    new_population = [
        max(population, key=fitness)
    ]

    while len(new_population) < 6:
        parent1, parent2 = random.choices(
            population,
            weights=scores,
            k=2
        )

        point = random.randint(1, 4)

        child = parent1[:point] + parent2[point:]

        child = ''.join(
            bit if random.random() > 0.05
            else str(1 - int(bit))
            for bit in child
        )

        new_population.append(child)

    population = new_population

best = max(population, key=fitness)

print("Best x =", decode(best))
print("Maximum f(x) =", fitness(best))

Best x = 31
Maximum f(x) = 961


In [11]:
import random

weights = [2, 3, 4, 5]
values = [3, 4, 5, 6]
capacity = 8

random.seed(11)

def total_weight(chromosome):
    return sum(
        weights[i]
        for i, bit in enumerate(chromosome)
        if bit == '1'
    )

def fitness(chromosome):
    weight = total_weight(chromosome)

    value = sum(
        values[i]
        for i, bit in enumerate(chromosome)
        if bit == '1'
    )

    return value if weight <= capacity else 0

population = ['0101'] + [
    ''.join(random.choice('01') for _ in range(4))
    for _ in range(7)
]

for _ in range(30):
    ranked = sorted(
        population,
        key=fitness,
        reverse=True
    )

    new_population = ranked[:2]

    while len(new_population) < 8:
        parent1, parent2 = random.sample(ranked[:4], 2)

        point = random.randint(1, 3)

        child = parent1[:point] + parent2[point:]

        child = ''.join(
            bit if random.random() > 0.10
            else str(1 - int(bit))
            for bit in child
        )

        new_population.append(child)

    population = new_population

best = max(population, key=fitness)

print("Best chromosome:", best)
print("Total weight:", total_weight(best))
print("Total value:", fitness(best))

Best chromosome: 0101
Total weight: 8
Total value: 10


In [12]:
import random

random.seed(12)

X = [-5 + i * 10 / 49 for i in range(50)]
y = [x * x + 3 * x + 2 for x in X]

def mean_squared_error(program):
    b, c = program

    predictions = [
        x * x + b * x + c
        for x in X
    ]

    return sum(
        (actual - predicted) ** 2
        for actual, predicted in zip(y, predictions)
    ) / len(X)

population = [
    (random.randint(-5, 5), random.randint(-5, 5))
    for _ in range(30)
]

for _ in range(100):
    population.sort(key=mean_squared_error)

    next_generation = population[:4]

    while len(next_generation) < 30:
        parent1, parent2 = random.sample(population[:10], 2)

        child = [parent1[0], parent2[1]]

        if random.random() < 0.30:
            child[0] = random.randint(-5, 5)

        if random.random() < 0.30:
            child[1] = random.randint(-5, 5)

        next_generation.append(tuple(child))

    population = next_generation

best = min(population, key=mean_squared_error)

b, c = best

print(f"Evolved expression: x^2 + {b}x + {c}")
print(f"Mean squared error: {mean_squared_error(best):.1f}")

Evolved expression: x^2 + 3x + 2
Mean squared error: 0.0


In [13]:
prior = 0.01

sensitivity = 0.99

false_positive_rate = 0.05

positive_evidence = (
    sensitivity * prior
    + false_positive_rate * (1 - prior)
)

posterior = (
    sensitivity * prior
) / positive_evidence

print(
    f"P(Disease | Positive Test) = {posterior:.4f}"
)

P(Disease | Positive Test) = 0.1667


In [14]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=0
)

model = GaussianNB()

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(
    f"Accuracy: {accuracy_score(y_test, predictions):.4f}"
)

print("Confusion matrix:")
print(confusion_matrix(y_test, predictions))

Accuracy: 1.0000
Confusion matrix:
[[13  0  0]
 [ 0 16  0]
 [ 0  0  9]]


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix

messages = [
    'win a free prize now',
    'claim your free cash prize',
    'limited offer win money',
    'free entry in contest',
    'urgent claim your reward',
    'meeting scheduled for tomorrow',
    'please send the project report',
    'are we meeting today',
    'lunch at noon',
    'your assignment is ready',
    'win cash now',
    'project meeting tomorrow',
    'free prize claim now',
    'please call me later',
    'urgent free offer'
]

labels = [
    'spam', 'spam', 'spam', 'spam', 'spam',
    'ham', 'ham', 'ham', 'ham', 'ham',
    'spam', 'ham', 'spam', 'ham', 'spam'
]

X_train, X_test, y_train, y_test = train_test_split(
    messages,
    labels,
    test_size=0.20,
    random_state=1,
    stratify=labels
)

vectorizer = CountVectorizer().fit(X_train)

X_train_vec = vectorizer.transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = MultinomialNB()

model.fit(X_train_vec, y_train)

predictions = model.predict(X_test_vec)

print(
    f"Accuracy: {accuracy_score(y_test, predictions):.2f}"
)

print("Confusion matrix (ham, spam):")

print(
    confusion_matrix(
        y_test,
        predictions,
        labels=['ham', 'spam']
    )
)

Accuracy: 1.00
Confusion matrix (ham, spam):
[[1 0]
 [0 2]]


In [18]:
from itertools import product

P_RAIN = 0.20
P_SPRINKLER = 0.40

p_wet = {
    (True, True): 0.99,
    (True, False): 0.90,
    (False, True): 0.90,
    (False, False): 0.01
}

joint_wet = []

for rain, sprinkler in product([False, True], repeat=2):

    p_rain = P_RAIN if rain else 1 - P_RAIN

    p_sprinkler = (
        P_SPRINKLER if sprinkler
        else 1 - P_SPRINKLER
    )

    joint = (
        p_rain *
        p_sprinkler *
        p_wet[(rain, sprinkler)]
    )

    joint_wet.append((rain, joint))

p_wet_total = sum(
    probability
    for _, probability in joint_wet
)

p_rain_and_wet = sum(
    probability
    for rain, probability in joint_wet
    if rain
)

posterior = p_rain_and_wet / p_wet_total

print(
    f"P(Rain=True | Wet=True) = {posterior:.4f}"
)

P(Rain=True | Wet=True) = 0.3900


In [19]:
from sklearn.datasets import make_blobs
from sklearn.mixture import GaussianMixture

X, y = make_blobs(
    n_samples=300,
    centers=3,
    cluster_std=0.7,
    random_state=1
)

model = GaussianMixture(
    n_components=3,
    random_state=1
)

model.fit(X)

labels = model.predict(X)

print("Converged:", model.converged_)
print("Number of samples clustered:", len(labels))

Converged: True
Number of samples clustered: 300


In [20]:
import numpy as np

data = np.array(
    [2, 4, 4, 4, 5, 5, 7, 9],
    dtype=float
)

mle_mean = np.mean(data)

mle_variance = np.var(data)

candidates = np.arange(0, 11, dtype=float)

sse = [
    np.sum((data - candidate) ** 2)
    for candidate in candidates
]

best_candidate = candidates[int(np.argmin(sse))]

print(f"MLE mean: {mle_mean:.2f}")

print(f"MLE variance: {mle_variance:.2f}")

print(
    f"Candidate minimising SSE: {best_candidate:.0f}"
)

MLE mean: 5.00
MLE variance: 4.00
Candidate minimising SSE: 5


In [21]:
import math

hypotheses = list(range(10))

target = lambda x: x >= 4

mistakes = 0

for x in range(10):

    votes = [
        x >= threshold
        for threshold in hypotheses
    ]

    majority_prediction = (
        sum(votes) > len(votes) / 2
    )

    true_label = target(x)

    if majority_prediction != true_label:
        mistakes += 1

    hypotheses = [
        threshold
        for threshold in hypotheses
        if (x >= threshold) == true_label
    ]

print("Mistakes made:", mistakes)

print(
    "Theoretical bound log2(|H|):",
    round(math.log2(10), 2)
)

Mistakes made: 1
Theoretical bound log2(|H|): 3.32


In [22]:
import math

def sample_complexity(
    hypothesis_count,
    epsilon,
    delta
):
    return math.ceil(
        (
            math.log(hypothesis_count)
            + math.log(1 / delta)
        ) / epsilon
    )

for size in [10, 100, 1000]:

    required = sample_complexity(
        size,
        epsilon=0.1,
        delta=0.05
    )

    print(
        f"|H|={size}: m >= {required} examples needed"
    )

|H|=10: m >= 53 examples needed
|H|=100: m >= 77 examples needed
|H|=1000: m >= 100 examples needed


In [23]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=1
)

model = KNeighborsClassifier(n_neighbors=5)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(
    f"Accuracy: {accuracy_score(y_test, predictions):.4f}"
)

print("First five predictions:", predictions[:5])

Accuracy: 0.9778
First five predictions: [0 1 1 0 2]


In [24]:
import numpy as np

from sklearn.neighbors import KNeighborsRegressor

X_train = np.array([
    [0],
    [1],
    [2],
    [3],
    [4]
], dtype=float)

y_train = np.array([0, 2, 4, 6, 8], dtype=float)

X_query = np.array([
    [1.5],
    [2.5]
])

model = KNeighborsRegressor(n_neighbors=2)

model.fit(X_train, y_train)

predictions = model.predict(X_query)

print("Predictions:", predictions)

Predictions: [3. 5.]


In [25]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=1
)

uniform_model = KNeighborsClassifier(
    n_neighbors=5,
    weights='uniform'
)

distance_model = KNeighborsClassifier(
    n_neighbors=5,
    weights='distance'
)

uniform_model.fit(X_train, y_train)

distance_model.fit(X_train, y_train)

print(
    f"Uniform-weighted accuracy: "
    f"{accuracy_score(y_test, uniform_model.predict(X_test)):.4f}"
)

print(
    f"Distance-weighted accuracy: "
    f"{accuracy_score(y_test, distance_model.predict(X_test)):.4f}"
)

Uniform-weighted accuracy: 0.9778
Distance-weighted accuracy: 0.9778


In [26]:
import numpy as np

def locally_weighted_regression(
    x_query,
    X,
    y,
    tau=1.0
):
    weights = np.exp(
        -((X - x_query) ** 2) / (2 * tau ** 2)
    )

    design = np.column_stack((
        np.ones(len(X)),
        X
    ))

    W = np.diag(weights)

    theta = (
        np.linalg.pinv(design.T @ W @ design)
        @ design.T @ W @ y
    )

    return np.array([1.0, x_query]) @ theta


X = np.array([0, 1, 2, 3, 4], dtype=float)

y = np.array([0, 1, 4, 9, 16], dtype=float)

queries = [0.5, 1.5, 2.5]

predictions = [
    locally_weighted_regression(x, X, y)
    for x in queries
]

print("Sample predictions:", np.round(predictions, 3))

Sample predictions: [0.623 3.123 7.123]


In [27]:
import numpy as np

from sklearn.linear_model import LogisticRegression

X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)

y = np.array([0, 1, 1, 0])

centres = X.copy()

sigma = 0.5

def rbf_transform(data, centres, width):
    return np.array([
        [
            np.exp(
                -np.sum((sample - centre) ** 2)
                / (2 * width ** 2)
            )
            for centre in centres
        ]
        for sample in data
    ])

X_rbf = rbf_transform(X, centres, sigma)

model = LogisticRegression(
    C=1000,
    solver='lbfgs',
    random_state=0
)

model.fit(X_rbf, y)

predictions = model.predict(X_rbf)

print("Predictions:", predictions)

print(
    "Correctly classified:",
    int(np.sum(predictions == y)),
    "of",
    len(y)
)

Predictions: [0 1 1 0]
Correctly classified: 4 of 4


In [28]:
case_library = [
    {
        'symptoms': {
            'fever': True,
            'cough': True,
            'fatigue': False
        },
        'diagnosis': 'Flu'
    },
    {
        'symptoms': {
            'fever': False,
            'cough': True,
            'fatigue': True
        },
        'diagnosis': 'Cold'
    },
    {
        'symptoms': {
            'fever': True,
            'cough': False,
            'fatigue': True
        },
        'diagnosis': 'Viral Infection'
    }
]

def similarity(new_case, old_case):
    return sum(
        new_case[key] == old_case.get(key)
        for key in new_case
    )

new_symptoms = {
    'fever': True,
    'cough': True,
    'fatigue': False
}

best_case = max(
    case_library,
    key=lambda case: similarity(
        new_symptoms,
        case['symptoms']
    )
)

print(
    "Retrieved diagnosis (Reuse):",
    best_case['diagnosis']
)

case_library.append({
    'symptoms': new_symptoms,
    'diagnosis': best_case['diagnosis']
})

print(
    "Case library size after Retain:",
    len(case_library)
)

Retrieved diagnosis (Reuse): Flu
Case library size after Retain: 4


In [29]:
import numpy as np

np.set_printoptions(
    precision=3,
    suppress=True
)

a = np.array([1, 2, 3, 4])

zeros = np.zeros((2, 3), dtype=int)

steps = np.arange(0, 10, 2)

points = np.linspace(0, 1, 5)

print(
    "a:",
    a,
    "shape:",
    a.shape,
    "dimensions:",
    a.ndim
)

print("zeros:\n", zeros)

print("arange:", steps)

print("linspace:", points)

x = np.array([1, 2, 3])

y = np.array([4, 5, 6])

print("x + y:", x + y)

print("x * y:", x * y)

print("dot product:", np.dot(x, y))

print("sqrt(x):", np.sqrt(x))

print("exp(x):", np.exp(x))

a: [1 2 3 4] shape: (4,) dimensions: 1
zeros:
 [[0 0 0]
 [0 0 0]]
arange: [0 2 4 6 8]
linspace: [0.   0.25 0.5  0.75 1.  ]
x + y: [5 7 9]
x * y: [ 4 10 18]
dot product: 32
sqrt(x): [1.    1.414 1.732]
exp(x): [ 2.718  7.389 20.086]


In [30]:
import numpy as np

a = np.arange(12).reshape(3, 4)

print("Array:\n", a)

print("a[1, 2]:", a[1, 2])

print("Column 1:", a[:, 1])

print("Elements greater than 5:", a[a > 5])

a[a % 2 == 0] = 0

print("After zeroing even values:\n", a)

Array:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
a[1, 2]: 6
Column 1: [1 5 9]
Elements greater than 5: [ 6  7  8  9 10 11]
After zeroing even values:
 [[ 0  1  0  3]
 [ 0  5  0  7]
 [ 0  9  0 11]]


In [31]:
import numpy as np

a = np.array([
    [1, 2, 3],
    [4, 5, 6]
])

print("Mean:", a.mean())

print("Sum along columns:", a.sum(axis=0))

print("Sum along rows:", a.sum(axis=1))

print(f"Standard deviation: {a.std():.3f}")

print("Minimum per row:", a.min(axis=1))

print("Maximum overall:", a.max())

Mean: 3.5
Sum along columns: [5 7 9]
Sum along rows: [ 6 15]
Standard deviation: 1.708
Minimum per row: [1 4]
Maximum overall: 6


In [32]:
import numpy as np

rng = np.random.default_rng(42)

np.set_printoptions(
    precision=3,
    suppress=True
)

uniform = rng.random((2, 3))

integers = rng.integers(0, 10, size=5)

gaussian = rng.normal(0, 1, size=5)

print("Uniform random 2x3:\n", uniform)

print("Random integers:", integers)

print("Gaussian samples:", gaussian)

n = 100000

points = rng.random((n, 2))

inside = np.sum(
    points[:, 0] ** 2 + points[:, 1] ** 2 <= 1
)

print(f"Estimated pi: {4 * inside / n:.5f}")

Uniform random 2x3:
 [[0.774 0.439 0.859]
 [0.697 0.094 0.976]]
Random integers: [7 7 7 7 5]
Gaussian samples: [-0.853  0.879  0.778  0.066  1.127]
Estimated pi: 3.15012


In [33]:
import numpy as np

a = np.array([
    [1, 2, 3],
    [4, 5, 6]
])

np.save('array.npy', a)

np.savetxt(
    'array.csv',
    a,
    delimiter=',',
    fmt='%d'
)

loaded_npy = np.load('array.npy')

loaded_csv = np.loadtxt(
    'array.csv',
    delimiter=',',
    dtype=int
)

print("Loaded from NPY:\n", loaded_npy)

print("Loaded from CSV:\n", loaded_csv)

Loaded from NPY:
 [[1 2 3]
 [4 5 6]]
Loaded from CSV:
 [[1 2 3]
 [4 5 6]]


In [34]:
import pandas as pd

students = pd.DataFrame({
    'name': ['A', 'B', 'C'],
    'marks': [80, 90, 70]
})

new_student = pd.DataFrame({
    'name': ['D'],
    'marks': [85]
})

combined = pd.concat(
    [students, new_student],
    ignore_index=True
)

print("Concatenated:")
print(combined.to_string(index=False))

grades = pd.DataFrame({
    'name': ['A', 'B', 'C'],
    'grade': ['A', 'A+', 'B']
})

merged = pd.merge(
    students,
    grades,
    on='name',
    how='inner'
)

print("Merged:")
print(merged.to_string(index=False))

Concatenated:
name  marks
   A     80
   B     90
   C     70
   D     85
Merged:
name  marks grade
   A     80     A
   B     90    A+
   C     70     B


In [35]:
from pathlib import Path
import pandas as pd

Path('students.csv').write_text(
    'name,marks,class\nA,78,I\nB,45,II\n',
    encoding='utf-8'
)

df = pd.read_csv('students.csv')

print("Loaded data:")
print(df.to_string(index=False))

df['pass'] = df['marks'] >= 50

df.to_csv(
    'students_processed.csv',
    index=False
)

print("Processed data:")
print(df.to_string(index=False))

Loaded data:
name  marks class
   A     78     I
   B     45    II
Processed data:
name  marks class  pass
   A     78     I  True
   B     45    II False


In [36]:
import pandas as pd

df = pd.DataFrame({
    'class': ['I', 'II', 'I', 'II', 'I', 'II'],
    'marks': [78, 85, 92, 66, 74, 88]
})

summary = df.groupby('class')['marks'].agg([
    'mean',
    'max',
    'min'
])

print(
    summary.to_string(
        float_format=lambda value: f'{value:.2f}'
    )
)

       mean  max  min
class                
I     81.33   92   74
II    79.67   88   66


In [37]:
import pandas as pd

df = pd.DataFrame({
    'name': ['A', 'B', 'C', 'D'],
    'class': ['I', 'II', 'I', 'II'],
    'marks': [78, 85, 92, 66]
})

filtered = df[
    (df['marks'] > 70) &
    (df['class'] == 'I')
]

sorted_df = df.sort_values(
    ['class', 'marks'],
    ascending=[True, False]
)

print("Filtered:")
print(filtered.to_string(index=False))

print("Sorted:")
print(sorted_df.to_string(index=False))

Filtered:
name class  marks
   A     I     78
   C     I     92
Sorted:
name class  marks
   C     I     92
   A     I     78
   B    II     85
   D    II     66


In [38]:
import matplotlib

matplotlib.use('Agg')

import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame({
    'name': ['A', 'B', 'C', 'D', 'E'],
    'marks': [78, 85, 92, 66, 74]
})

df['marks'].plot(
    kind='hist',
    title='Marks Distribution',
    bins=5
)

plt.xlabel('Marks')
plt.tight_layout()
plt.savefig('marks_hist.png')
plt.close()

df.plot(
    x='name',
    y='marks',
    kind='bar',
    title='Marks by Student',
    legend=False
)

plt.ylabel('Marks')
plt.tight_layout()
plt.savefig('marks_bar.png')
plt.close()

print("Plots saved: marks_hist.png, marks_bar.png")

Plots saved: marks_hist.png, marks_bar.png


In [39]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    'name': ['A', 'B', 'B', 'C', None],
    'marks': [78, np.nan, 85, 92, 66]
})

print("Missing values:")
print(df.isna().sum().to_string())

filled = df.fillna({
    'name': 'Unknown',
    'marks': df['marks'].mean()
})

print("After filling missing values:")

print(
    filled.to_string(
        index=False,
        formatters={'marks': lambda x: f'{x:.2f}'}
    )
)

with_duplicate = pd.concat(
    [filled, filled.iloc[[2]]],
    ignore_index=True
)

clean = with_duplicate.drop_duplicates()

print("After removing duplicates:")

print(
    clean.to_string(
        index=False,
        formatters={'marks': lambda x: f'{x:.2f}'}
    )
)

Missing values:
name     1
marks    1
After filling missing values:
   name marks
      A 78.00
      B 80.25
      B 85.00
      C 92.00
Unknown 66.00
After removing duplicates:
   name marks
      A 78.00
      B 80.25
      B 85.00
      C 92.00
Unknown 66.00


In [40]:
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

iris = load_iris(as_frame=True)

df = iris.frame.dropna().copy()

X = df[iris.feature_names]

y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=1
)

scaler = StandardScaler().fit(X_train)

X_train = scaler.transform(X_train)

X_test = scaler.transform(X_test)

model = KNeighborsClassifier(n_neighbors=5)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(
    f"Pipeline accuracy: {accuracy_score(y_test, predictions):.4f}"
)

print(
    "First five predicted species:",
    iris.target_names[predictions[:5]]
)

Pipeline accuracy: 0.9556
First five predicted species: ['setosa' 'versicolor' 'versicolor' 'setosa' 'virginica']


In [41]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=1
)

model = DecisionTreeClassifier(random_state=1)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Confusion matrix:")
print(confusion_matrix(y_test, predictions))

print(
    f"Accuracy: {accuracy_score(y_test, predictions):.4f}"
)

print(
    f"Precision (macro): "
    f"{precision_score(y_test, predictions, average='macro'):.4f}"
)

print(
    f"Recall (macro): "
    f"{recall_score(y_test, predictions, average='macro'):.4f}"
)

print(
    f"F1-score (macro): "
    f"{f1_score(y_test, predictions, average='macro'):.4f}"
)

Confusion matrix:
[[14  0  0]
 [ 0 17  1]
 [ 0  1 12]]
Accuracy: 0.9556
Precision (macro): 0.9558
Recall (macro): 0.9558
F1-score (macro): 0.9558


In [42]:
import matplotlib

matplotlib.use('Agg')

import matplotlib.pyplot as plt
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

iris = load_iris()

df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

df['target'] = iris.target

print("Dataset rows:", len(df))

print("Dataset columns:", len(df.columns))

X_train, X_test, y_train, y_test = train_test_split(
    df.iloc[:, :-1],
    df['target'],
    test_size=0.30,
    random_state=1
)

models = {
    'Decision Tree': DecisionTreeClassifier(
        random_state=1
    ),

    'KNN (k=5)': KNeighborsClassifier(
        n_neighbors=5
    )
}

results = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    results[name] = accuracy_score(
        y_test,
        model.predict(X_test)
    )

for name, score in results.items():
    print(f"{name}: {score:.4f}")

plt.bar(
    results.keys(),
    results.values()
)

plt.ylabel('Accuracy')

plt.title('Model Comparison - Iris Dataset')

plt.tight_layout()

plt.savefig('model_comparison.png', dpi=150)

print("Chart saved as model_comparison.png")

Dataset rows: 150
Dataset columns: 5
Decision Tree: 0.9556
KNN (k=5): 0.9778
Chart saved as model_comparison.png
